# 00c — Saraga Carnatic: Data Preparation & Transcription

**Thesis context:** This notebook processes the Carnatic subset of the Saraga 1.5 dataset.
Carnatic classical music differs from Hindustani in its rhythmic organisation (tala system),
melodic structures (raga with different characteristic phrases), and ensemble interaction
patterns (primary vocal with violin, mridangam, and ghatam).

**Metadata note:** Unlike the Hindustani subset, many Carnatic JSON files have empty `raaga`
arrays. This appears to be a known metadata completeness issue in Saraga Carnatic 1.5.
Where `raaga` is empty, the raga name is inferred from the piece title or concert folder name.
This limitation is documented and affects a majority of the 249 curated tracks.

**Track selection:** The Saraga Carnatic collection contains 991 MP3 files — 249 are full
concert mixes (listed in file_paths.csv); the remainder are separated audio stems (vocal,
violin, mridangam). We use only the full mixes. 60 tracks are selected across the 27
available concert folders for performer diversity.

**Output:**
- `data/processed/carnatic/midi/` — 60 transcribed MIDI files
- `data/metadata/carnatic_tracks.csv` — per-track metadata

**Time estimate:** ~5–8 hours on CPU.


In [9]:
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.12.0


In [10]:
import sys
from pathlib import Path

# Locate project root regardless of where Jupyter was launched from.
# Searches upward for PROGRESS.md — the root marker.
_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [11]:
import pandas as pd
import json
import shutil
import time

from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

RAW_DIR  = PROJECT_ROOT / "datasets" / "indian_classical" / "saraga1.5_carnatic"
OUT_MIDI = PROJECT_ROOT / "data" / "processed" / "carnatic" / "midi"
META_DIR = PROJECT_ROOT / "data" / "metadata"

OUT_MIDI.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

## 1. Scan tracks from file_paths.csv

In [12]:
fpc = pd.read_csv(RAW_DIR / "file_paths.csv", index_col=0)
print(f"file_paths.csv entries (full mixes): {len(fpc)}")
fpc.head()

file_paths.csv entries (full mixes): 249


,filepath,mbid
0,Sumitra Nitin at Arkay by Sumitra Nitin/Doraku...,6d4e58d1-e565-4987-b7a5-c63a4e9d3f90
1,Sumitra Nitin at Arkay by Sumitra Nitin/Ganamu...,d99f623e-8004-4e1a-a31c-5f760a018498
2,Sumitra Nitin at Arkay by Sumitra Nitin/Chidam...,eaf6b71a-8baf-4df2-bf70-0586428c7faa
3,Sumitra Nitin at Arkay by Sumitra Nitin/Vandal...,e3fe24f3-292a-4a96-97fd-bfd1f681394c
4,Sumitra Nitin at Arkay by Sumitra Nitin/Thiruv...,7cbd62c6-2304-459a-8d7a-98647abac6d2


In [13]:
def find_carnatic_audio(raw_dir, filepath_stem):
    """Locate the Carnatic audio file; Carnatic files may use single .mp3 extension."""
    for ext in [".mp3", ".mp3.mp3"]:
        p = raw_dir / (filepath_stem + ext)
        if p.exists():
            return p
    return None


def parse_carnatic_json(json_path):
    """
    Extract raga/taal/form from a Saraga Carnatic JSON file.

    Key difference from Hindustani: the field is 'raaga' (singular),
    and the raga object often has only 'name' (Unicode transliteration),
    not 'common_name'. Many files have an empty 'raaga' list.
    """
    try:
        meta = json.load(open(json_path, encoding="utf-8"))
        raaga_list = meta.get("raaga", [])
        taala_list = meta.get("taala", [])
        form_list  = meta.get("form", [])
        return {
            "title"    : meta.get("title", ""),
            "mbid"     : meta.get("mbid", ""),
            # Prefer 'common_name'; fall back to 'name' (Unicode)
            "raga"     : (raaga_list[0].get("common_name") or raaga_list[0].get("name", ""))
                         if raaga_list else None,
            "taal"     : (taala_list[0].get("common_name") or taala_list[0].get("name", ""))
                         if taala_list else None,
            "form"     : (form_list[0].get("name", "")) if form_list else None,
            "length_ms": meta.get("length", None),
        }
    except Exception as e:
        return {"error": str(e)}


records = []
for _, row in fpc.iterrows():
    fp_stem = row["filepath"]
    audio   = find_carnatic_audio(RAW_DIR, fp_stem)
    json_p  = RAW_DIR / (fp_stem + ".json")
    meta    = parse_carnatic_json(json_p) if json_p.exists() else {}

    raga     = meta.get("raga")
    meta_src = "json"
    if not raga:
        # Fallback: use the piece title (middle component of path)
        parts    = fp_stem.split("/")
        raga_raw = parts[1] if len(parts) >= 2 else parts[-1]
        raga     = raga_raw.strip() or "Unknown"
        meta_src = "title_fallback"

    records.append({
        "filepath_stem"   : fp_stem,
        "audio_path"      : str(audio) if audio else None,
        "title"           : meta.get("title", ""),
        "mbid"            : row.get("mbid", ""),
        "raga"            : raga,
        "taal"            : meta.get("taal", ""),
        "form"            : meta.get("form", ""),
        "length_ms"       : meta.get("length_ms", None),
        "metadata_source" : meta_src,
        "concert_folder"  : fp_stem.split("/")[0],
    })

tracks_df = pd.DataFrame(records)
print(f"Total tracks: {len(tracks_df)}")
print(f"\nMetadata source breakdown:")
print(tracks_df["metadata_source"].value_counts().to_string())
print(f"\nTracks with audio: {tracks_df['audio_path'].notna().sum()}")
print(f"Concert folders  : {tracks_df['concert_folder'].nunique()}")

Total tracks: 249

Metadata source breakdown:
metadata_source
title_fallback    162
json               87

Tracks with audio: 149
Concert folders  : 26


## 2. Select 60 tracks across concert folders

We prioritise tracks with JSON-sourced raga labels and sample to cover all 27 concert
folders (i.e. performers), ensuring performer diversity in the selected subset.


In [14]:
N = 60
available = tracks_df[tracks_df["audio_path"].notna()].copy()
print(f"Tracks with audio available: {len(available)}")

# Stratify by concert folder (performer) for diversity
sampled = (
    available
    .groupby("concert_folder", group_keys=False)
    .apply(lambda x: x.sample(
        min(len(x), max(1, round(N * len(x) / len(available)))),
        random_state=42
    ))
)

if len(sampled) > N:
    sampled = sampled.sample(N, random_state=42)
elif len(sampled) < N:
    remaining = available[~available["filepath_stem"].isin(sampled["filepath_stem"])]
    topup = remaining.sample(min(N - len(sampled), len(remaining)), random_state=42)
    sampled = pd.concat([sampled, topup])

sampled = sampled.reset_index(drop=True)
print(f"Selected: {len(sampled)} tracks across {sampled['concert_folder'].nunique()} performers")
print(f"\nMetadata source in selection:")
print(sampled["metadata_source"].value_counts().to_string())

Tracks with audio available: 149
Selected: 60 tracks across 17 performers

Metadata source in selection:
metadata_source
json              36
title_fallback    24


/var/folders/88/d6f4kns57fz2gzfzhd0dktfw0000gn/T/ipykernel_48700/1861842272.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  available


In [15]:
sampled.to_csv(META_DIR / "carnatic_tracks.csv", index=False)
print("Selection saved → data/metadata/carnatic_tracks.csv")

Selection saved → data/metadata/carnatic_tracks.csv


## 3. Basic-Pitch transcription

Same configuration as Hindustani (TF backend, ICASSP_2022 model). Idempotent —
re-run safely after interruption.


In [16]:
midi_paths = []
failed     = []

for i, row in sampled.iterrows():
    audio_path = Path(row["audio_path"])
    raga_safe  = str(row["raga"]).replace(" ", "_").replace("/", "_")[:30]
    out_name   = f"carnatic_{i:03d}_{raga_safe}.mid"
    out_path   = OUT_MIDI / out_name

    if out_path.exists():
        print(f"[{i+1:3d}/{len(sampled)}] SKIP  {out_name}")
        midi_paths.append(str(out_path))
        continue

    print(f"[{i+1:3d}/{len(sampled)}] Transcribing: {audio_path.name[:60]} ...", end="", flush=True)
    t0 = time.time()
    try:
        _, midi_data, _ = predict(str(audio_path), ICASSP_2022_MODEL_PATH)
        midi_data.write(str(out_path))
        elapsed = time.time() - t0
        print(f" {elapsed/60:.1f} min")
        midi_paths.append(str(out_path))
    except Exception as e:
        print(f" FAILED: {e}")
        failed.append({"index": i, "audio": str(audio_path), "error": str(e)})
        midi_paths.append(None)

print(f"\n=== Transcription complete ===")
print(f"Succeeded : {sum(p is not None for p in midi_paths)}")
print(f"Failed    : {len(failed)}")

[  1/60] SKIP  carnatic_000_Shloka_Sri_Ramachandra_Shrita_.mid
[  2/60] SKIP  carnatic_001_Nera_Nammiti.mid
[  3/60] SKIP  carnatic_002_shanmukhapriya.mid
[  4/60] SKIP  carnatic_003_sindhubhairavi.mid
[  5/60] SKIP  carnatic_004_sahana.mid
[  6/60] SKIP  carnatic_005_kamavardani.mid
[  7/60] SKIP  carnatic_006_madyamavati.mid
[  8/60] SKIP  carnatic_007_padi.mid
[  9/60] Transcribing: Karuna Nidhi Illalo.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_carnatic/Cherthala Ranganatha Sharma at Arkay by Cherthala Ranganatha Sharma/Karuna Nidhi Illalo/Karuna Nidhi Illalo.mp3.mp3...


2026-06-22 12:11:09.105307: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor '72352' with dtype float and shape [36,1,256]
	 [[{{node 72352}}]]


 4.4 min
[ 10/60] Transcribing: Rama Rama Guna Seema.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_carnatic/Cherthala Ranganatha Sharma at Arkay by Cherthala Ranganatha Sharma/Rama Rama Guna Seema/Rama Rama Guna Seema.mp3.mp3...


2026-06-22 12:15:41.230731: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor '82178' with dtype float and shape [36,1,256]
	 [[{{node 82178}}]]


 7.4 min
[ 11/60] Transcribing: Jaya Jaya Padmanabhanujese.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_carnatic/Dr S Sundar at Arkay by S Sundar/Jaya Jaya Padmanabhanujese/Jaya Jaya Padmanabhanujese.mp3.mp3...


...

 3.8 min
[ 59/60] Transcribing: Kanakangaka Guha.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_carnatic/Vidya Subramanian at Arkay by Vidya Subramanian/Kanakangaka Guha/Kanakangaka Guha.mp3.mp3...


2026-06-22 13:42:25.733919: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor '485492' with dtype float and shape [36,1,256]
	 [[{{node 485492}}]]


 0.4 min
[ 60/60] Transcribing: Vinakayunna Della.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_carnatic/Vidya Subramanian at Arkay by Vidya Subramanian/Vinakayunna Della/Vinakayunna Della.mp3.mp3...


2026-06-22 13:42:49.137853: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor '491986' with dtype float and shape [36,1,256]
	 [[{{node 491986}}]]


 2.0 min

=== Transcription complete ===
Succeeded : 60
Failed    : 0


## 4. Update metadata and verify

In [17]:
sampled = sampled.copy()
sampled["midi_path"] = midi_paths
sampled.to_csv(META_DIR / "carnatic_tracks.csv", index=False)

midi_files = list(OUT_MIDI.glob("*.mid"))
print(f"MIDI files in output dir    : {len(midi_files)}")
print(f"Tracks with MIDI path       : {sampled['midi_path'].notna().sum()}")
print(f"Performer (concert) coverage: {sampled['concert_folder'].nunique()}")
print("\n✓ Carnatic preparation complete.")

MIDI files in output dir    : 60
Tracks with MIDI path       : 60
Performer (concert) coverage: 17

✓ Carnatic preparation complete.
